In [ ]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import os
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
# base_dir = "/content/gdrive/MyDrive/Final Project"
sys.path.append(base_dir)
from Models.Baseline_Model.GPT2_Baseline import GPT2_Baseline
from Models.Baseline_Model.train_Baseline import train_loop, estimate_loss
from Models.Configs import TrainConfig, BaselineConfig
from Datasets.DataLoader import CombinedBinDataLoader


device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


In [ ]:
#Load the hyperparameters, scheduler, etc... required for training
model_config = BaselineConfig()
train_config = TrainConfig()
eff_batch_size = train_config.batch_per_iter * train_config.grad_acc_factor
tokens_per_step = eff_batch_size * model_config.block_size


model = GPT2_Baseline(model_config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = train_config.make_optimizer(model)
scheduler = train_config.make_scheduler(optimizer)
scaler = train_config.make_scaler()

get_lr = train_config.get_lr
torch.set_float32_matmul_precision('high')


In [ ]:
#Load the combined_bin dataset from drive and place it in Colab's files
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin',
    train_config.batch_per_iter,
    model_config.block_size,
    train_config,
    seed=42
)

In [ ]:
#Actually train the model
torch.cuda.empty_cache()
history = train_loop(model,
                     optimizer,
                     scheduler,
                     scaler,
                     device,
                     train_loader,
                     val_loader,
                     train_config,
                     model_config)

In [ ]:
message = model.infer("""One, two, three, """, 100, .8, 50)
print(message)

In [ ]:
#Save the model's state after training
path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_baseline_step_3623.pt"
torch.save({
            "step": 3623 ,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history
        }, path)

In [ ]:
loss_fwd, ppl = estimate_loss(model, val_loader, device, 160)
print(f"Val FWD: {loss_fwd:.4f} | PPL: {ppl:.2f}")

In [ ]:
#Test model Accuracy
